# Day 3 — Dividend-adjusted European Black–Scholes

This notebook makes each pricing step visible for learning. It is not imported by production code. Inputs use annual decimal rates and volatility, time in years, and prices per share.

For spot $S$, strike $K$, time $T$, volatility $\sigma$, continuously compounded risk-free rate $r$, and continuous dividend yield $q$:

$$d_1 = \frac{\ln(S/K)+(r-q+\sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2=d_1-\sigma\sqrt{T}$$
$$C=Se^{-qT}N(d_1)-Ke^{-rT}N(d_2), \quad P=Ke^{-rT}N(-d_2)-Se^{-qT}N(-d_1)$$

In [ ]:
import numpy as np
from scipy.special import ndtr

S, K, T, sigma, r, q = 100.0, 100.0, 1.0, 0.20, 0.05, 0.02
d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)
call = S * np.exp(-q*T) * ndtr(d1) - K * np.exp(-r*T) * ndtr(d2)
put = K * np.exp(-r*T) * ndtr(-d2) - S * np.exp(-q*T) * ndtr(-d1)
print(f'd1={d1:.4f}, d2={d2:.4f}, call={call:.4f}, put={put:.4f}')
assert np.isclose(call, 9.227006, atol=1e-6)
assert np.isclose(put, 6.330081, atol=1e-6)

In [ ]:
# Put–call parity is a strong consistency check.
parity_left = call - put
parity_right = S * np.exp(-q*T) - K * np.exp(-r*T)
print(f'C-P={parity_left:.8f}; discounted S-K={parity_right:.8f}')
assert np.isclose(parity_left, parity_right)

In [ ]:
# Direction experiment: call value as spot and volatility change.
def call_price(spot, vol):
    x1 = (np.log(spot/K) + (r-q+0.5*vol**2)*T) / (vol*np.sqrt(T))
    x2 = x1 - vol*np.sqrt(T)
    return spot*np.exp(-q*T)*ndtr(x1) - K*np.exp(-r*T)*ndtr(x2)

spots = np.array([90.0, 100.0, 110.0])
spot_experiment = np.array([call_price(s, 0.20) for s in spots])
vol_experiment = np.array([call_price(100.0, v) for v in [0.10, 0.20, 0.30]])
print('spot experiment:', spot_experiment)
print('volatility experiment:', vol_experiment)
assert np.all(np.diff(spot_experiment) > 0)
assert np.all(np.diff(vol_experiment) > 0)

## Interpretation and limitations

These are theoretical model values, not market prices. The model assumes European exercise, lognormal price dynamics, constant volatility/rates/dividend yield, frictionless trading, and no arbitrage. At expiry, model value becomes payoff. At zero volatility before expiry, use the discounted deterministic payoff rather than dividing by zero.